In [1]:
import math
import csv
import statistics

FRAC_BITS = 14

INPUT_INT_BITS = 5          
INTERNAL_INT_BITS = 4       
OUTPUT_INT_BITS = 2         

INPUT_WIDTH = INPUT_INT_BITS + FRAC_BITS
INTERNAL_WIDTH = INTERNAL_INT_BITS + FRAC_BITS
OUTPUT_WIDTH = OUTPUT_INT_BITS + FRAC_BITS

LSB = 2 ** (-FRAC_BITS)

MAX_ITER = 16
FULL_ITER = 14
DEFAULT_MIN_ITER = 6

RANGE_REDUCE_THRESHOLD_FLOAT = 1.0
SATURATION_THRESHOLD_FLOAT = 5.2       

ERROR_THRESHOLD_LSB = 1
QUALIFY_THRESHOLD_COUNTS = ERROR_THRESHOLD_LSB

X_SWEEP_RANGE = 8.0
N_DENSE_NEAR_ZERO = 400
N_LOG_SPACED_TAIL = 150
N_MAGNITUDE_BINS = 8

def to_fixed(value_float, frac_bits=FRAC_BITS):
    """Float -> fixed-point int, rounding to nearest."""
    return round(value_float * (1 << frac_bits))


def from_fixed(value_int, frac_bits=FRAC_BITS):
    """Fixed-point int -> float, for reporting/comparison only."""
    return value_int / (1 << frac_bits)


def saturate(value_int, width):
    """Clamp to a signed two's-complement range of `width` bits. Returns (clamped, overflow_flag)."""
    lo = -(1 << (width - 1))
    hi = (1 << (width - 1)) - 1
    if value_int > hi:
        return hi, True
    if value_int < lo:
        return lo, True
    return value_int, False


def round_div(numerator, denominator):
    """Integer division, round-to-nearest, correct for negative operands."""
    if denominator == 0:
        return 0
    sign = -1 if (numerator < 0) ^ (denominator < 0) else 1
    n, d = abs(numerator), abs(denominator)
    return sign * ((n + d // 2) // d)


def fx_mul(a, b, frac_bits=FRAC_BITS):
    """Fixed-point multiply with round-to-nearest rescale."""
    return round_div(a * b, 1 << frac_bits)


def fx_div(a, b, frac_bits=FRAC_BITS):
    """Fixed-point divide with round-to-nearest rescale."""
    return round_div(a << frac_bits, b)


def ashr(value_int, shift, round_mode=False):
    """
    Arithmetic right shift by `shift` bits — this IS the '* 2^-i' CORDIC
    micro-op in hardware. Python's >> on an int already performs a correct
    arithmetic (sign-preserving, floor-toward -inf) shift, matching a
    synthesizable barrel/fixed shifter on a two's-complement signal.

    round_mode=True adds a rounding bit before shifting (round-to-nearest)
    -- this costs an extra adder per shifter in real hardware, so it is
    a real design choice, not free. Both modes are compared in main().
    """
    if shift <= 0:
        return value_int
    if round_mode and shift > 0:
        value_int = value_int + (1 << (shift - 1))
    return value_int >> shift




def cordic_iteration_indices(n_iterations):
    seq = []
    i = 1
    count = 0
    while count < n_iterations:
        seq.append(i)
        count += 1
        if i in (4, 13, 40) and count < n_iterations:
            seq.append(i)
            count += 1
        i += 1
    return seq




_MAX_I_NEEDED = 20
ATANH_ROM = {i: to_fixed(math.atanh(2.0 ** (-i)), FRAC_BITS) for i in range(1, _MAX_I_NEEDED)}




def hyperbolic_cordic_tanh_steps_fixed(z0_fixed, n_iterations, round_mode=False):
    """
    Pure integer hyperbolic CORDIC rotation mode.
    Returns (estimates, overflow_flag): estimates[n-1] = tanh estimate
    (as an INTERNAL-format fixed-point int) after n cumulative iterations.
    """
    x = 1 << FRAC_BITS   
    y = 0
    z = z0_fixed
    estimates = []
    overflow = False
    seq = cordic_iteration_indices(n_iterations)

    for i in seq:
        d = 1 if z >= 0 else -1
        shifted_y = ashr(y, i, round_mode)
        shifted_x = ashr(x, i, round_mode)

        x_new = x + d * shifted_y
        y_new = y + d * shifted_x
        z_new = z - d * ATANH_ROM[i]

        x_new, ov1 = saturate(x_new, INTERNAL_WIDTH)
        y_new, ov2 = saturate(y_new, INTERNAL_WIDTH)
        z_new, ov3 = saturate(z_new, INTERNAL_WIDTH)
        overflow = overflow or ov1 or ov2 or ov3

        x, y, z = x_new, y_new, z_new
        tanh_est = fx_div(y, x) if x != 0 else 0
        estimates.append(tanh_est)

    return estimates, overflow


def tanh_estimates_fixed(x_input_fixed, max_iter=MAX_ITER, round_mode=False):
    """
    Full integer pipeline: saturation check -> range reduction (integer
    halving) -> CORDIC -> integer doubling-formula reconstruction.
    x_input_fixed is an INPUT-format fixed-point int.
    Returns (estimates[list of OUTPUT-format ints], k_shifts, overflow_flag).
    """
    sat_threshold_fixed = to_fixed(SATURATION_THRESHOLD_FLOAT)
    if abs(x_input_fixed) >= sat_threshold_fixed:
        one_minus_lsb = (1 << FRAC_BITS) - 1
        sat_val = one_minus_lsb if x_input_fixed > 0 else -one_minus_lsb - 1
        sat_val, _ = saturate(sat_val, OUTPUT_WIDTH)
        return [sat_val] * max_iter, 0, False

    threshold_fixed = to_fixed(RANGE_REDUCE_THRESHOLD_FLOAT)
    val = x_input_fixed
    k = 0
    while abs(val) > threshold_fixed:
        val = ashr(val, 1, round_mode)
        k += 1

    raw_estimates, overflow = hyperbolic_cordic_tanh_steps_fixed(val, max_iter, round_mode)

    final = []
    for est in raw_estimates:
        t = est
        for _ in range(k):
            t_sq = fx_mul(t, t)
            denom = (1 << FRAC_BITS) + t_sq
            numer = 2 * t
            t = fx_div(numer, denom)
        t_out, ov = saturate(t, OUTPUT_WIDTH)
        overflow = overflow or ov
        final.append(t_out)

    return final, k, overflow


def sigmoid_estimates_fixed(x_input_fixed, max_iter=MAX_ITER, round_mode=False):
    """sigmoid(x) = 0.5*(1+tanh(x/2)), computed in pure integer arithmetic."""
    x_half = ashr(x_input_fixed, 1, round_mode)
    tanh_ests, k, overflow = tanh_estimates_fixed(x_half, max_iter, round_mode)
    sig_ests = []
    for t in tanh_ests:
        s = ashr((1 << FRAC_BITS) + t, 1, round_mode)
        s, ov = saturate(s, OUTPUT_WIDTH)
        overflow = overflow or ov
        sig_ests.append(s)
    return sig_ests, k, overflow


def build_sweep():
    xs = set()
    for i in range(N_DENSE_NEAR_ZERO + 1):
        v = -1.0 + 2.0 * i / N_DENSE_NEAR_ZERO
        xs.add(round(v, 8))
    for i in range(1, N_LOG_SPACED_TAIL + 1):
        frac = i / N_LOG_SPACED_TAIL
        v = 1.0 + (X_SWEEP_RANGE - 1.0) * (10 ** (frac * 2) - 1) / 99.0
        xs.add(round(v, 8))
        xs.add(round(-v, 8))
    return sorted(xs)


def run_sweep(round_mode):
    xs = build_sweep()
    rows = []
    overflow_count = 0
    bin_edges = [i * X_SWEEP_RANGE / N_MAGNITUDE_BINS for i in range(N_MAGNITUDE_BINS + 1)]
    bin_required_iter = {b: 0 for b in range(N_MAGNITUDE_BINS)}

    for x in xs:
        ref_tanh = math.tanh(x)
        x_fixed = to_fixed(x, FRAC_BITS)

        estimates, k, overflow = tanh_estimates_fixed(x_fixed, MAX_ITER, round_mode)
        if overflow:
            overflow_count += 1

        full_result_fixed = estimates[FULL_ITER - 1]
        full_result_float = from_fixed(full_result_fixed)

        
        
        residual_counts = [abs(e - full_result_fixed) for e in estimates]
        ref_error_float = [abs(from_fixed(e) - ref_tanh) for e in estimates]

        qualified_n = None
        for n in range(1, FULL_ITER + 1):
            if all(residual_counts[m - 1] <= QUALIFY_THRESHOLD_COUNTS for m in range(n, FULL_ITER + 1)):
                qualified_n = n
                break
        if qualified_n is None:
            qualified_n = FULL_ITER

        ax = abs(x)
        b = min(int(ax / X_SWEEP_RANGE * N_MAGNITUDE_BINS), N_MAGNITUDE_BINS - 1)
        bin_required_iter[b] = max(bin_required_iter[b], qualified_n)

        rows.append({
            "x": x,
            "x_fixed_hex": hex(x_fixed & ((1 << INPUT_WIDTH) - 1)),
            "range_reduce_shifts": k,
            "ref_tanh_double_precision": ref_tanh,
            "full_iter_result_fixed": full_result_fixed,
            "full_iter_result_float": full_result_float,
            "ref_error_at_full_iter": ref_error_float[FULL_ITER - 1],
            "qualified_min_iterations": qualified_n,
            "residual_lsb_at_min_iter_6": residual_counts[DEFAULT_MIN_ITER - 1],
            "ref_error_at_min_iter_6": ref_error_float[DEFAULT_MIN_ITER - 1],
            "overflow": overflow,
        })

    return xs, rows, overflow_count, bin_edges, bin_required_iter


def summarize(mode_name, xs, rows, overflow_count, bin_edges, bin_required_iter):
    qualified_counts = [r["qualified_min_iterations"] for r in rows]
    default6_ok = sum(1 for r in rows if r["residual_lsb_at_min_iter_6"] <= QUALIFY_THRESHOLD_COUNTS)
    default6_ok_pct = 100.0 * default6_ok / len(rows)
    ref_errors_at_full = [r["ref_error_at_full_iter"] for r in rows]
    worst_ref_error_at_full = max(ref_errors_at_full)

    lines = []
    lines.append(f"=== Mode: {mode_name} shift rounding ===")
    lines.append(f"Sweep size: {len(xs)} points, x in [-{X_SWEEP_RANGE}, {X_SWEEP_RANGE}]")
    lines.append(f"Formats (FRAC_BITS={FRAC_BITS}): "
                 f"INPUT={INPUT_WIDTH}b, INTERNAL={INTERNAL_WIDTH}b, OUTPUT={OUTPUT_WIDTH}b")
    lines.append(f"Overflow (saturation) events observed: {overflow_count} / {len(rows)} "
                 f"{'(width recommendation OK)' if overflow_count == 0 else '(WIDTH TOO NARROW -- widen INTERNAL_INT_BITS)'}")
    lines.append(f"Min/Mean/Max qualified iterations: {min(qualified_counts)} / "
                 f"{statistics.mean(qualified_counts):.2f} / {max(qualified_counts)}")
    lines.append(f"MIN_ITER={DEFAULT_MIN_ITER} sufficient for {default6_ok_pct:.1f}% of inputs "
                 f"(exact-LSB-count residual check)")
    lines.append(f"Worst-case error vs true double-precision reference at FULL_ITER={FULL_ITER}: "
                 f"{worst_ref_error_at_full:.6e} ({worst_ref_error_at_full / LSB:.2f} LSB)")
    lines.append("LUT magnitude-class qualified-iteration table:")
    for b in range(N_MAGNITUDE_BINS):
        lo, hi = bin_edges[b], bin_edges[b + 1]
        lines.append(f"  |x| in [{lo:.2f}, {hi:.2f}): {bin_required_iter[b]} iterations")
    lines.append("")
    return "\n".join(lines)


def main():
    all_summary = []
    all_summary.append("=== TRUE Bit-Accurate (integer-only) CORDIC Model — Summary ===\n")

    csv_rows = None
    for round_mode, name in [(False, "TRUNCATE (no rounding on shifts -- cheapest hardware)"),
                              (True, "ROUND-TO-NEAREST (extra adder per shifter)")]:
        xs, rows, overflow_count, bin_edges, bin_required_iter = run_sweep(round_mode)
        all_summary.append(summarize(name, xs, rows, overflow_count, bin_edges, bin_required_iter))
        if not round_mode:
            csv_rows = rows

    all_summary.append(
        "--- Interpretation ---\n"
        "If TRUNCATE and ROUND modes show similar worst-case error and mean qualified-iteration\n"
        "count, the extra rounding adder on each shifter is not earning its area -- use TRUNCATE.\n"
        "If ROUND meaningfully reduces the worst-case error or lets you lower FULL_ITER by 1-2\n"
        "iterations, the rounding logic may be worth the area tradeoff. Compare the two blocks above.\n"
    )

    summary_text = "\n".join(all_summary)
    print(summary_text)
    with open("summary_fixed.txt", "w") as f:
        f.write(summary_text)

    with open("test_vectors_fixed.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(csv_rows[0].keys()))
        writer.writeheader()
        writer.writerows(csv_rows)


if __name__ == "__main__":
    main()

=== TRUE Bit-Accurate (integer-only) CORDIC Model — Summary ===

=== Mode: TRUNCATE (no rounding on shifts -- cheapest hardware) shift rounding ===
Sweep size: 701 points, x in [-8.0, 8.0]
Formats (FRAC_BITS=14): INPUT=19b, INTERNAL=18b, OUTPUT=16b
Overflow (saturation) events observed: 0 / 701 (width recommendation OK)
Min/Mean/Max qualified iterations: 1 / 11.67 / 14
MIN_ITER=6 sufficient for 8.0% of inputs (exact-LSB-count residual check)
Worst-case error vs true double-precision reference at FULL_ITER=14: 5.103393e-04 (8.36 LSB)
LUT magnitude-class qualified-iteration table:
  |x| in [0.00, 1.00): 14 iterations
  |x| in [1.00, 2.00): 14 iterations
  |x| in [2.00, 3.00): 13 iterations
  |x| in [3.00, 4.00): 10 iterations
  |x| in [4.00, 5.00): 8 iterations
  |x| in [5.00, 6.00): 2 iterations
  |x| in [6.00, 7.00): 1 iterations
  |x| in [7.00, 8.00): 1 iterations

=== Mode: ROUND-TO-NEAREST (extra adder per shifter) shift rounding ===
Sweep size: 701 points, x in [-8.0, 8.0]
Formats 